# Credit Risk Scoring v2 - Mejora versión anterior
## Contexto 
Creamos una herramienta que nos sirve para detectar posibles defaults con respecto a los clientes solventes. Es importante, tener detectados a los clientes defaults para evitar posibles pérdidas de capital para una entidad financiera. Si rechazamos a un buen cliente, es un menor riesgo, pero también es crítico por posibles fugas de clientes con el que podríamos generar beneficio a medio/largo plazo.

## Dataset
El conjunto de datos se extrae de un dataset público del German Credit Data en OpenML.


# Imports

In [35]:
import pandas as pd
from sklearn.datasets import fetch_openml
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

# Carga de datos

In [36]:
credit = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
X = credit.data
y = credit.target

### Visualización de la dimensiones del dataset

In [37]:
print("Dimensión de X:",X.shape)
print("Dimensión de y:",y.shape)

Dimensión de X: (1000, 20)
Dimensión de y: (1000,)


### Vista previa y balance del target

In [38]:
print(X.head())
y.value_counts(normalize=True)  #sin normalize=True nos da valores absolutos, pero con este parámetro te devuelve la proporción

  checking_status  duration                  credit_history  \
0              <0         6  critical/other existing credit   
1        0<=X<200        48                   existing paid   
2     no checking        12  critical/other existing credit   
3              <0        42                   existing paid   
4              <0        24              delayed previously   

               purpose  credit_amount    savings_status employment  \
0             radio/tv           1169  no known savings        >=7   
1             radio/tv           5951              <100     1<=X<4   
2            education           2096              <100     4<=X<7   
3  furniture/equipment           7882              <100     4<=X<7   
4              new car           4870              <100     1<=X<4   

   installment_commitment     personal_status other_parties  residence_since  \
0                       4         male single          none                4   
1                       2  female div/de

class
good    0.7
bad     0.3
Name: proportion, dtype: float64

### Balance del target

- El resultado de los targets 70/30 significa que existe un desbalanceo moderado de clientes. 
- Si de 1000 clientes tenemos 300 morosos, según los importes puede afectar a la solvencia de la entidad. 
- El accuracy es engañosos, ya que tenga un porcentaje de acierto de un 70% no significa que no haya riesgo, porque no tener en cuenta el 30% restante puede implicar que exista un riesgo de default loss que se debe tener en cuenta. 


## Selección de variables

In [39]:
X.columns

Index(['checking_status', 'duration', 'credit_history', 'purpose',
       'credit_amount', 'savings_status', 'employment',
       'installment_commitment', 'personal_status', 'other_parties',
       'residence_since', 'property_magnitude', 'age', 'other_payment_plans',
       'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone',
       'foreign_worker'],
      dtype='object')

He escogido las siguientes variables:

1. duration - duración del préstamo.
2. credit_amount - importe solicitado.
3. credit_history - historial crediticio.
4. checking_status - estado cuenta corriente.
5. savings_status - nivel de ahorros.
6. employment - antigüedad laboral.

In [40]:
X_sel = X[['duration', 'credit_amount', 'credit_history', 'checking_status', 'savings_status', 'employment']]

In [41]:
print(X_sel.shape)
X_sel.head()

(1000, 6)


,duration,credit_amount,credit_history,checking_status,savings_status,employment
0,6,1169,critical/other existing credit,<0,no known savings,>=7
1,48,5951,existing paid,0<=X<200,<100,1<=X<4
2,12,2096,critical/other existing credit,no checking,<100,4<=X<7
3,42,7882,existing paid,<0,<100,4<=X<7
4,24,4870,delayed previously,<0,<100,1<=X<4


### One Hot Encoding

In [42]:
X_encoded = pd.get_dummies(X_sel, drop_first=True)

In [43]:
print(X_encoded.shape)
X_encoded.head()

(1000, 17)


,duration,credit_amount,credit_history_critical/other existing credit,credit_history_delayed previously,credit_history_existing paid,credit_history_no credits/all paid,checking_status_<0,checking_status_>=200,checking_status_no checking,savings_status_500<=X<1000,savings_status_<100,savings_status_>=1000,savings_status_no known savings,employment_4<=X<7,employment_<1,employment_>=7,employment_unemployed
0,6,1169,True,False,False,False,True,False,False,False,False,False,True,False,False,True,False
1,48,5951,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False
2,12,2096,True,False,False,False,False,False,True,False,True,False,False,True,False,False,False
3,42,7882,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False
4,24,4870,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False


Hemos aplicado One Hot Encoding a las 6 variables seleccionadas (X_sel). Las 2 variables numéricas se quedan igual, pero las variables categóricas, se dividen en varias columnas borrando una opción y si se cumple se marca como True y si no, se marca como False.

## Train/Test/Split
parámetros que utilizamos: 
* test_size=0.2 -> reserva el 20% para test, 80% para train.
* random_state=42 -> para fijar la aleatoriedad. El 42 es una convención.
* stratify=y -> como los datos están desbalanceados, es obligatorio utilizarlo. 

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

In [45]:
print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(800, 17)
(200, 17)
class
good    0.7
bad     0.3
Name: proportion, dtype: float64
class
good    0.7
bad     0.3
Name: proportion, dtype: float64


## Escalado de variables numéricas


In [46]:
cols_numericas = ['duration', 'credit_amount']
scaler = StandardScaler()
X_train[cols_numericas] = scaler.fit_transform(X_train[cols_numericas])
X_test[cols_numericas] = scaler.transform(X_test[cols_numericas])
print(X_train[cols_numericas].head())

     duration  credit_amount
675  0.755149       0.485384
703  0.755149      -0.246578
12  -0.726746      -0.584573
845  0.014201       0.285331
795 -0.973728      -0.319522


## Baseline: Logistic Regression

In [47]:
# Implementación Logistic Regression

modelo_lr = LogisticRegression(max_iter=1000, class_weight='balanced')
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)
print(y_pred_lr)

['good' 'bad' 'bad' 'good' 'bad' 'good' 'good' 'bad' 'bad' 'good' 'good'
 'good' 'bad' 'bad' 'good' 'bad' 'bad' 'good' 'good' 'good' 'good' 'good'
 'good' 'good' 'good' 'good' 'good' 'bad' 'good' 'bad' 'good' 'good'
 'good' 'good' 'bad' 'bad' 'bad' 'bad' 'good' 'bad' 'good' 'good' 'good'
 'bad' 'bad' 'bad' 'bad' 'good' 'bad' 'good' 'good' 'bad' 'bad' 'good'
 'good' 'good' 'good' 'bad' 'good' 'good' 'good' 'good' 'good' 'good'
 'bad' 'good' 'good' 'bad' 'bad' 'good' 'good' 'bad' 'good' 'bad' 'good'
 'bad' 'good' 'bad' 'bad' 'bad' 'bad' 'bad' 'good' 'bad' 'bad' 'good'
 'bad' 'good' 'good' 'good' 'bad' 'good' 'bad' 'good' 'good' 'bad' 'bad'
 'bad' 'bad' 'bad' 'good' 'bad' 'good' 'bad' 'good' 'good' 'good' 'good'
 'bad' 'good' 'good' 'good' 'good' 'bad' 'good' 'good' 'good' 'good' 'bad'
 'good' 'bad' 'good' 'good' 'good' 'good' 'bad' 'good' 'good' 'good' 'bad'
 'good' 'bad' 'good' 'good' 'bad' 'good' 'bad' 'bad' 'bad' 'good' 'good'
 'good' 'good' 'bad' 'good' 'bad' 'bad' 'good' 'bad' 'bad'

Se tuvo que escalar las variables duration y credit_amount, porque los valores tienen una media muy grande de valores. Y se ha aplicado fit_transform en train, transform en test, por

## Evaluacion: Logistic Regression

In [48]:
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

[[45 15]
 [47 93]]
              precision    recall  f1-score   support

         bad       0.49      0.75      0.59        60
        good       0.86      0.66      0.75       140

    accuracy                           0.69       200
   macro avg       0.68      0.71      0.67       200
weighted avg       0.75      0.69      0.70       200



Como sklearn ordena alfabéticamente arriba tienes bad y abajo good. La matriz de confusión la interpretamos: 
* 45 fueron etiquetados como morosos correctamente.
* 15 fueron etiquetados como buen cliente, pero al final resultaron ser morosos. Punto principal a reforzar (FN), porque se deberían detectar correctamente o disminuir la probabilidad de que suceda.
* 47 fueron etiquetados como morosos y se le negó el préstamo, pero eran buenos clientes (coste de oportunidad) - este punto se debe reforzar porque es negativo para el prestamista, pero como segundo punto.
* 93 préstamos aprobados correctamente.

* recall 0.75% nos está indicando que de 60 morosos (45 fueron detectados y 15 no).
* precision 0.49 marcó 92 como morosos (45 reales y 47 falsos). 

* El modelo ha marcado como morosos a buenos clientes porque con class_weight='balanced' todo aquel que tiene algún parámetro de moroso, lo marca directamente como moroso, sin analizar más datos. Esto conlleva a que marque a clientes buenos como morosos. Para un banco, por ejemplo, que los buenos clientes son también importantes para no tener fugas hacia otras entidades por estos rechazos que al final se traduce en menos fondos y menos con lo que trabajar para generar como empresa y los morosos no detectados podrían ser una gran pérdida de capital.
 
* Si queremos ser un banco conservador, en época de crisis financiera, se rechaza mucho, pero si es una época de captación de cuota de mercado, prefieren asumir el riesgo de aceptar a posibles morosos por no perder clientes.

## Random Forest

In [49]:
# Creacion modelo random forest
modelo_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

# Entrenamiento modelo
modelo_rf.fit(X_train, y_train)

# Predicción
y_pred_rf = modelo_rf.predict(X_test)

## Evaluacion: Random Forest

In [50]:
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[ 31  29]
 [ 21 119]]
              precision    recall  f1-score   support

         bad       0.60      0.52      0.55        60
        good       0.80      0.85      0.83       140

    accuracy                           0.75       200
   macro avg       0.70      0.68      0.69       200
weighted avg       0.74      0.75      0.74       200



El modelo Random Forest tiene mejor accuracy que el modelo Logistic Regression, pero para el análisis de morosidad si nos fijamos en la detección de morosos, RF es peor ya que ha dado préstamos a 29 morosos con respecto a los 15 que nos daría LR. Esto se debe a que Random Forest por mucho que le indiques balanced es más conservador a la hora de etiquetar a un moroso y eso implica que su recall sobre la clase 'bad' disminuye. Esto implica que hay un mayor riesgo de que conceda préstamos a un moroso por falta de detección. 

## Threshold tuning: validation set

In [51]:
# Creacion de un set de validacion sobre el train. 800 filas, le damos el 25% a validation (200) y el resto se queda en train.

X_train_final, X_val, y_train_final, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

In [54]:
print(X_train_final.shape)
print(X_val.shape)

(600, 17)
(200, 17)


# Reentrenamiento de Random Forest finetuneando el umbral

In [58]:
# Reentreno sobre las 600 filas y calculo de probabilidades
modelo_rf_v2 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
modelo_rf_v2.fit(X_train_final, y_train_final)

# Probabilidad sobre validation
probas_val = modelo_rf_v2.predict_proba(X_val)

print(probas_val[:5])


[[0.65 0.35]
 [0.7  0.3 ]
 [0.28 0.72]
 [0.46 0.54]
 [0.01 0.99]]


In [59]:
# Extraccion columna bad
probas_bad = probas_val[:,0]

print(probas_bad[:5])

[0.65 0.7  0.28 0.46 0.01]


## Threshold tuning: probar con umbral 0.35

In [61]:
umbral = 0.35
y_pred_umbral = np.where(probas_bad >= umbral, 'bad', 'good')
print(confusion_matrix(y_val, y_pred_umbral))
print(classification_report(y_val, y_pred_umbral))

[[ 35  25]
 [ 32 108]]
              precision    recall  f1-score   support

         bad       0.52      0.58      0.55        60
        good       0.81      0.77      0.79       140

    accuracy                           0.71       200
   macro avg       0.67      0.68      0.67       200
weighted avg       0.73      0.71      0.72       200



Bajar el umbral hemos conseguido un incremento en el recall de bad y con ello conseguimos cazar más morosos.